# ParEGO implementation fix and fixed-budget performance

How does the official fix change optimization performance, and how does corrected native ParEGO compare with HVPI, NDS and Random? Both experiments use VLMOP2. This notebook reads existing evidence without fitting models or collecting timings.

In [1]:
from pathlib import Path
import json
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
stats = json.loads((root / 'results/upgrade/statistics.json').read_text())
assert stats['runs_audited'] == 120
print('Retained evidence: 120 VLMOP2 trajectories / 6000 evaluations')


Retained evidence: 120 VLMOP2 trajectories / 6000 evaluations


## Experiment 1 - Effect of the ParEGO implementation fix

Both operators run inside PHYSBO 3.2.1 with paired initial points, GP settings and an independently logged weight stream. The official correction belongs to PHYSBO maintainers. This isolates the reduction; it does not reproduce complete earlier releases.

In [2]:
for method in ['ParEGO-max', 'ParEGO-min']:
    row = next(r for r in stats['summary'] if r['block'] == 'version' and r['method'] == method and r['evaluations'] == 50)
    print(method, 'final raw HV:', round(row['mean'], 4), '95% interval:', (round(row['low'], 4), round(row['high'], 4)))
pair = stats['paired_comparisons'][0]
print('Paired min-minus-max raw-HV difference:', round(pair['mean'], 4), '95% interval:', (round(pair['low'], 4), round(pair['high'], 4)))


ParEGO-max final raw HV: 0.1588 95% interval: (0.1341, 0.183)
ParEGO-min final raw HV: 0.2762 95% interval: (0.2629, 0.2876)
Paired min-minus-max raw-HV difference: 0.1174 95% interval: (0.0891, 0.1469)


![Controlled operator effect](../figures/upgrade/version_comparison.png)

The corrected min operator improves observed mean final quality in this controlled comparison.

## Experiment 2 - Corrected ParEGO under a fixed evaluation budget

Native ParEGO, HVPI, NDS and Random share ten initial observations and a 50-evaluation cap. The native ParEGO weight mechanism differs from the diagnostic paired stream; the two conditions are not pooled.

In [3]:
methods = ['ParEGO', 'HVPI', 'NDS', 'Random']
for method in methods:
    row = next(r for r in stats['summary'] if r['block'] == 'multi' and r['method'] == method and r['evaluations'] == 50)
    print(method, 'final raw HV:', round(row['mean'], 4), '95% interval:', (round(row['low'], 4), round(row['high'], 4)),
          '90% target hits:', f"{round(row['target_hit_rate']*20)}/20", 'decision seconds:', round(row['mean_decision_s'], 5))


ParEGO final raw HV: 0.2713 95% interval: (0.2472, 0.291) 90% target hits: 4/20 decision seconds: 2.71841
HVPI final raw HV: 0.327 95% interval: (0.326, 0.3278) 90% target hits: 20/20 decision seconds: 6.94829
NDS final raw HV: 0.2516 95% interval: (0.2195, 0.2804) 90% target hits: 6/20 decision seconds: 2.64981
Random final raw HV: 0.1922 95% interval: (0.1795, 0.2043) 90% target hits: 0/20 decision seconds: 0.00045


![Fixed-budget strategies](../figures/upgrade/multi_quality.png)

At this fixed budget native ParEGO has a lower observed mean final HV than HVPI and a higher mean than NDS and Random. This statement is conditional on the current task and settings; it is not a new pairwise significance claim.

In [4]:
for n in [10, 20, 50]:
    means = {m: round(next(r['mean'] for r in stats['summary'] if r['block'] == 'multi' and r['method'] == m and r['evaluations'] == n), 4) for m in methods}
    print('Queries:', n, 'mean raw HV:', means)
for target in [.8, .9, .95]:
    hits = {m: next(r['successes'] for r in stats['target_sensitivity'] if r['block'] == 'multi' and r['method'] == m and r['target'] == target) for m in methods}
    print('Grid-HV target:', target, 'hits out of 20:', hits)


Queries: 10 mean raw HV: {'ParEGO': 0.0795, 'HVPI': 0.0795, 'NDS': 0.0795, 'Random': 0.0795}
Queries: 20 mean raw HV: {'ParEGO': 0.1455, 'HVPI': 0.2555, 'NDS': 0.1456, 'Random': 0.129}
Queries: 50 mean raw HV: {'ParEGO': 0.2713, 'HVPI': 0.327, 'NDS': 0.2516, 'Random': 0.1922}
Grid-HV target: 0.8 hits out of 20: {'ParEGO': 15, 'HVPI': 20, 'NDS': 13, 'Random': 0}
Grid-HV target: 0.9 hits out of 20: {'ParEGO': 4, 'HVPI': 20, 'NDS': 6, 'Random': 0}
Grid-HV target: 0.95 hits out of 20: {'ParEGO': 0, 'HVPI': 20, 'NDS': 0, 'Random': 0}


## Evidence boundary

These are existing summaries of the same retained runs: 40 controlled operator trajectories and 80 native strategy trajectories. Original observations and timings remain unchanged. Full-grid HV and targets are used only for offline evaluation. Algorithm overhead is reported separately from quality; no physical experiment is performed.